## 基于混合 GPD 的个股尾部风险动态配置策略

**目标：** 每个交易日，预测股票池里每只股票发生极端亏损的风险。风险低的股票多配，风险高的股票少配；当整个股票池的风险都偏高时，降低总仓位、多留现金。

整个流程分为四步：

```
13 个因子  ──神经网络──▶  5 个分布参数  ──公式──▶  风险评分 R  ──配置规则──▶  个股权重 + 现金比例
```

---

### 第 1 步：神经网络输出五个参数

每个交易日 $t$，对股票 $i$ 取 13 个因子 $X_{i,t}$：10 个原始因子，加上 3 个历史因子（worst252、e63、echo_q）。所有因子只用到 $t$ 日及之前的数据。神经网络据此输出五个参数：

| 参数 | 含义 | 是否随股票、日期变化 | 取值范围 |
|---|---|---|---|
| $\xi_b$, $\beta_b$ | **基础分量**的形状和尺度，描述"普通的"超阈值亏损 | 是 | $0.01 < \xi_b < 0.35$ |
| $\xi_h$, $\beta_h$ | **重尾分量**的形状和尺度，描述"非常极端"的亏损 | 否，所有股票共用同一组 | $0.35 < \xi_h < 0.80$ |
| $\pi$ | 重尾分量所占的权重 | 是 | $0.03 \le \pi < 1$ |

- **形状参数 $\xi$：** 越大，尾部越"厚"，越容易出现特别大的亏损。
- **尺度参数 $\beta$：** 越大，亏损的整体幅度越大。
- **$\pi$ 的下限 0.03：** 保证重尾分量永远不会被完全关掉，任何股票都保留一定的极端风险。

---

### 第 2 步：五个参数描述的亏损分布

记日亏损为 $L$（即负的日收益），$u$ 是模型训练时使用的阈值。模型只描述亏损已经超过 $u$ 的情况，超出部分记为 $y = L - u$。

$y$ 的分布是两个 GPD（广义帕累托分布）的混合。它的生存函数，也就是"$y$ 超过某个值的概率"，为

$$
S(y) = (1-\pi)\, S_{\mathrm{GPD}}(y;\ \xi_b, \beta_b) \;+\; \pi\, S_{\mathrm{GPD}}(y;\ \xi_h, \beta_h),
$$

其中单个 GPD 的生存函数是

$$
S_{\mathrm{GPD}}(y;\ \xi, \beta) = \left(1 + \frac{\xi\, y}{\beta}\right)^{-1/\xi}.
$$

**直观理解：** 亏损一旦超过 $u$，有 $1-\pi$ 的可能按"普通"分布继续发展，有 $\pi$ 的可能按"极端"分布发展。

---

### 第 3 步：风险评分 $R$

设灾难性亏损阈值 $d = 7\%$。令 $a = d - u$，它是灾难线在 $y$ 上对应的位置。风险评分定义为**超过灾难线之后的期望超额亏损**：

$$
R_{i,t} = \mathbb{E}\left[(y - a)^{+}\right]
= (1-\pi_{i,t})\, R_{\mathrm{GPD}}(\xi_b, \beta_b) \;+\; \pi_{i,t}\, R_{\mathrm{GPD}}(\xi_h, \beta_h),
$$

其中 $(x)^{+} = \max(x, 0)$。每个分量的贡献是

$$
R_{\mathrm{GPD}}(\xi, \beta)
= \left(1 + \frac{\xi\, a}{\beta}\right)^{-1/\xi}
\times \frac{\beta + \xi\, a}{1 - \xi}.
$$

这个公式由两个因子相乘：
- **第一个因子：** 超过灾难线的概率。
- **第二个因子：** 一旦超过，平均还会再亏多少（Expected Shortfall）。

所以 $R$ 越大，这只股票的灾难性亏损风险越高。

---

### 第 4 步：个股权重和总仓位

**(a) 相对权重：风险倒数加权。** 风险越低，权重越高：

$$
v_{i,t} = \frac{(R_{i,t} + \delta)^{-\gamma}}{\sum_{j=1}^{N} (R_{j,t} + \delta)^{-\gamma}}.
$$

- **$\gamma$：** 控制对风险差异的敏感程度。越大，越偏向低风险股票；$\gamma = 0$ 就是等权。
- **$\delta$：** 一个很小的正数，防止 $R$ 接近 0 时数值不稳定。

**(b) 满仓风险：** 如果按 $v$ 满仓买入，组合的风险是

$$
B_t = \sum_{i=1}^{N} v_{i,t}\, R_{i,t}.
$$

**(c) 总仓位：** 和预先设定的风险预算 $R_{\mathrm{target}}$ 比较：

$$
c_t = \min\left(1,\ \frac{R_{\mathrm{target}}}{B_t}\right).
$$

满仓风险没有超过预算就满仓；超过了就按比例减仓。

**(d) 最终持仓：**

$$
w_{i,t} = c_t\, v_{i,t}, \qquad w_{\mathrm{cash},t} = 1 - c_t.
$$

---

### 局限

模型描述的是"亏损已经超过 $u$"这一前提下的分布，没有预测亏损超过 $u$ 这件事本身发生的概率。因此 $R_{i,t}$ 是**条件**尾部风险评分，可以用来比较不同股票、不同日期的相对风险，但不是完整的"每日预期灾难损失"。

### 从两参数模型到五参数模型：策略改了什么

**一句话总结：** 只有风险评分 $R$ 的计算方法变了。拿到 $R$ 之后，权重和仓位的规则（第 4 步）完全不变。

---

#### 改动 1：$R$ 的公式

| | 风险评分 |
|---|---|
| 旧模型（2 个参数） | $R = R_{\mathrm{GPD}}(\xi, \beta)$ |
| 新模型（5 个参数） | $R = (1-\pi)\, R_{\mathrm{GPD}}(\xi_b, \beta_b) + \pi\, R_{\mathrm{GPD}}(\xi_h, \beta_h)$ |

**为什么可以直接按 $\pi$ 加权相加？** 期望超额亏损可以写成生存函数的积分：

$$
R = \mathbb{E}\left[(y - a)^{+}\right] = \int_a^{\infty} S(y)\, dy.
$$

混合分布的 $S(y)$ 本身就是两个分量按 $1-\pi$ 和 $\pi$ 的加权和，积分可以拆开，所以 $R$ 也是两个分量的 $R$ 的加权和。每个分量直接套用原来的闭式公式即可，不需要数值积分。

**两种看起来合理、但其实是错的做法：**
1. **把五个参数"平均"成一组 $(\xi, \beta)$，再代入旧公式。** 两个 GPD 混合以后不再是一个 GPD，不存在这样一组等价的参数。
2. **先算混合后的"超过概率"和混合后的"平均额外损失"，再相乘。** 混合后的平均额外损失应该按"各分量超过灾难线的概率"加权，而不是按 $\pi$ 加权，很容易算错。直接把两个分量的 $R$ 相加，就绕开了这个问题。

---

#### 改动 2：重尾分量对 $R$ 的影响比 $\pi$ 看起来大

$\xi_h$ 可以接近 0.8，这时"平均额外损失"里的 $\frac{1}{1-\xi_h}$ 能放大到 5 倍。所以即使 $\pi$ 只有百分之几，重尾分量也可能贡献 $R$ 的一大半。

又因为 $\xi_h$ 和 $\beta_h$ 是所有股票共用的常数，重尾那一项可以写成 $\pi_{i,t} \times K$，其中 $K$ 对所有股票、所有日期都相同。由此得到两个结论：
- **股票之间的风险排序** 主要由 $\pi$ 和基础分量的 $(\xi_b, \beta_b)$ 决定。$\pi$ 越大的股票，分到的权重越少。
- **每只股票的 $R$ 都有下限 $0.03K$**，因为 $\pi \ge 0.03$。$R$ 不会接近 0，所以 $\delta$ 基本不再起作用，也不会出现某只股票权重特别大的情况。

想看每只股票的 $R$ 里有多少来自重尾分量，可以调用 `gpd_mixture_allocation(..., return_details=True)`，查看返回的 `R_base` 和 `R_heavy`。

---

#### 改动 3：阈值 $u$ 从模型读取

旧函数里 `u=0.03` 是写死的默认值。现在 $u$ 从 `preprocess.json` 的 `threshold_u_loss` 读出（即 `model.u`），作为必填参数传入，保证和训练时用的阈值一致。同时要求 $u < d$，否则灾难线落在阈值以下，公式没有意义。

---

#### 改动 4：输入格式和参数检查

- **输入：** 从 `(n, 2)` 数组改为 `(n, 5)` 数组，或带五个参数列的 DataFrame。
- **检查：**
  - $0 < \xi_b, \xi_h < 1$：$\xi \ge 1$ 时期望亏损是无穷大。
  - $\beta_b, \beta_h > 0$。
  - $0 \le \pi \le 1$。

---

#### 如何确认改对了

下面的"公式自检"单元格做了两项检查，不需要模型和数据：
1. **闭式公式 vs 数值积分：** 两者的相对误差在 $10^{-9}$ 量级。
2. **退化检查：** 令 $\pi = 0$，混合分布退化成单个 GPD $(\xi_b, \beta_b)$，新函数的输出和原来的两参数函数完全相同。

---

#### 代码中对应的函数

| 函数 | 作用 |
|---|---|
| `gpd_expected_excess` | 单个 GPD 的 $R_{\mathrm{GPD}}(\xi, \beta)$ |
| `mixture_tail_risk` | 五参数混合分布的 $R$，同时返回两个分量各自的贡献 |
| `allocate_from_risk` | 第 4 步：$R \to v \to B \to c \to w$，和旧版本相同 |
| `gpd_mixture_allocation` | 上面几步合起来：五个参数（和可选的 $p_u$）$\to$ 个股权重和现金比例 |

`gpd_expected_excess`、`mixture_tail_risk` 在 `tail_model.py` 里，`allocate_from_risk` 在 `strategy_core.py` 里，和 QuantConnect 用的是同一份代码。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# 整个项目共用的三个文件（都在仓库根目录）：
#   factor_core.py    13 个因子的计算
#   tail_model.py     两段模型的前向传播和风险指标（numpy，不需要 torch）
#   strategy_core.py  配置规则 allocate_from_risk
from tail_model import TwoStageTailModel, PARAM_COLS

# 神经网络参数的训练结果所在的文件夹（gpd_model.pt、pu_model.pt、preprocess.json）
ARTIFACT_DIR = Path("artifacts")

In [ ]:
# ==========================================
# 2-3. 数据预处理 + 神经网络前向传播：统一使用 tail_model.py
# ==========================================
# 整个项目只有这一份推理代码：训练脚本（训练后自动核对和 PyTorch 输出一致）、evaluate_tail.py、
# gpd_predict.py 和 QuantConnect 的 main.py 用的都是它。
# 预处理和训练时完全相同：缺失值填训练集中位数 -> 用训练集均值、标准差做 z-score -> 截断到 ±5；
# BatchNorm 使用训练得到的滑动统计量。

model = TwoStageTailModel.from_artifacts(ARTIFACT_DIR)
print(f"u = {model.u:.4%}, pi_min = {model.pi_min}, p_u 网络: {model.has_pu}")

# 用法（df = 含 13 个原始因子列的 DataFrame，未标准化）：
#   out = model.predict(df)               # xi_b, beta_b, xi_h, beta_h, pi, p_u
#   R   = model.risk_score(df, d=0.07)    # 风险评分 = p_u * E[(L-d)^+ | L>u]
#   model.var(df, 0.01), model.es(df, 0.01), model.tail_prob(df, 0.10)

In [ ]:
# 策略函数
# 风险评分的公式来自 tail_model.py，配置规则来自 strategy_core.py，与 QuantConnect 共用同一份代码。
#   gpd_expected_excess(xi, beta, a) = E[(Y - a)^+] = S(a) * (beta + xi*a) / (1 - xi)
#   mixture_tail_risk(params, d, u)  = (1 - pi) * R_base + pi * R_heavy，返回 (R, base_part, heavy_part)
#   allocate_from_risk(R, R_target, gamma, delta) -> weights, cash, v, B, c
from tail_model import gpd_expected_excess, mixture_tail_risk
from strategy_core import allocate_from_risk


def _as_param_array(gpd_params):
    """DataFrame（含 PARAM_COLS 列）或 (n, 5) 数组 -> (n, 5) 数组，并检查参数范围。"""
    if isinstance(gpd_params, pd.DataFrame):
        P = gpd_params[PARAM_COLS].to_numpy(dtype=float)
    else:
        P = np.asarray(gpd_params, dtype=float)
    if P.ndim != 2 or P.shape[1] != 5 or len(P) == 0:
        raise ValueError("gpd_params must have shape (n, 5): [xi_b, beta_b, xi_h, beta_h, pi].")
    if not np.all(np.isfinite(P)):
        raise ValueError("GPD parameters must be finite.")
    xi_b, beta_b, xi_h, beta_h, pi = P.T
    if np.any((xi_b <= 0) | (xi_b >= 1) | (xi_h <= 0) | (xi_h >= 1)):
        raise ValueError("Require 0 < xi_b, xi_h < 1 (otherwise the expected excess loss is infinite).")
    if np.any(beta_b <= 0) or np.any(beta_h <= 0):
        raise ValueError("Require beta_b, beta_h > 0.")
    if np.any((pi < 0) | (pi > 1)):
        raise ValueError("Require 0 <= pi <= 1.")
    return P


def gpd_mixture_allocation(gpd_params, R_target, u, gamma=1.0, delta=1e-6, d=0.07, p_u=None,
                           return_details=False):
    """
    gpd_params: (n, 5) 数组或 DataFrame = [xi_b, beta_b, xi_h, beta_h, pi]
    R_target:   目标灾难风险预算
    u:          模型的 GPD 阈值（必须用 model.u，和训练时一致）
    gamma:      相对权重对风险差异的敏感度
    delta:      数值稳定用的小正数
    d:          灾难性亏损阈值（默认 7%）
    p_u:        第一段网络的跌破概率 P(L > u | x)；给出时 R = p_u * E[(L-d)^+ | L>u]（推荐），
                None 时只用条件部分

    返回 weights（占总资产的比例）, cash；return_details=True 时再返回一个诊断字典。
    """
    if not (0 <= u < d):
        raise ValueError("Invalid thresholds (need 0 <= u < d).")
    P = _as_param_array(gpd_params)
    R, base_part, heavy_part = mixture_tail_risk(P, d, u)
    if p_u is not None:
        pu = np.asarray(p_u, dtype=float)
        R, base_part, heavy_part = pu * R, pu * base_part, pu * heavy_part
    weights, cash, v, B, c = allocate_from_risk(R, R_target, gamma, delta)
    if return_details:
        return weights, cash, dict(R=R, R_base=base_part, R_heavy=heavy_part, v=v, B=B, c=c)
    return weights, cash


# 等价写法（直接从原始因子出发）：
#   R = model.risk_score(df, d=0.07)          # 已乘上 p_u
#   weights, cash, v, B, c = allocate_from_risk(R, R_target, gamma, delta)

### 回测：在 QuantConnect 上进行

这个 notebook 只负责说明策略和检查公式。回测放在 QuantConnect 上做，代码在 `quantconnect/` 文件夹里：

| 文件 | 作用 |
|---|---|
| `quantconnect/main.py` | QuantConnect 算法：每天计算 13 个因子 → 两段模型 → 风险评分 → 调仓 |
| `quantconnect/export_model_for_qc.py` | 把 `artifacts/` 里的模型导出成 `gpd_mixture_model.json`（训练脚本训练后也会自动导出） |

**整个项目共用同一套代码**，QuantConnect 项目里需要原样加入仓库根目录的这三个文件：

| 文件 | 内容 | 使用者 |
|---|---|---|
| `factor_core.py` | 13 个因子的唯一实现 | `features.py`（训练数据）、`quantconnect/main.py` |
| `tail_model.py` | 两段模型的前向传播、S、VaR、ES、风险评分 | 训练脚本、`evaluate_tail.py`、`gpd_predict.py`、`quantconnect/main.py`、本 notebook |
| `strategy_core.py` | 配置规则 `allocate_from_risk` | `quantconnect/main.py`、本 notebook |

修改公式时只改这几个文件，训练、检验和回测会自动保持一致。